**dayofmonth():**
- `Returns day of month (1–31)`

**dayofyear():**
- `Returns day number in year (1–365/366)`.
- `considers leap year automatically`.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, current_date, date_format, dayofweek, dayofmonth, dayofyear

In [0]:
# define data
data = [['2023-01-01', 10],
        ['2023-04-02', 20],
        ['2023-04-08', 22],
        ['2023-04-15', 14],
        ['2023-04-17', 12],
        ['2023-05-21', 15],
        ['2023-05-23', 30],
        ['2023-10-26', 45],
        ['2023-10-28', 32],
        ['2023-10-29', 47],
        ['2023-12-31', 50]]
  
# define column names
columns = ['date', 'sales']
  
# create dataframe using data and column names
df_dow = spark.createDataFrame(data, columns) \
        .withColumn('date', col('date').cast('date')) \
        .withColumn('sales', col('sales').cast('integer'))

display(df_dow)
df_dow.printSchema()

date,sales
2023-01-01,10
2023-04-02,20
2023-04-08,22
2023-04-15,14
2023-04-17,12
2023-05-21,15
2023-05-23,30
2023-10-26,45
2023-10-28,32
2023-10-29,47


root
 |-- date: date (nullable = true)
 |-- sales: integer (nullable = true)



##### 1) Basic Example

In [0]:
df_wmd_01 = df_dow \
    .withColumn('day_name', F.date_format('date', 'EEEE')) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("day_of_month", dayofmonth(col("date"))) \
    .withColumn("day_of_year", dayofyear(col("date")))

display(df_wmd_01)
df_wmd_01.printSchema()

date,sales,day_name,day_of_week,day_of_month,day_of_year
2023-01-01,10,Sunday,1,1,1
2023-04-02,20,Sunday,1,2,92
2023-04-08,22,Saturday,7,8,98
2023-04-15,14,Saturday,7,15,105
2023-04-17,12,Monday,2,17,107
2023-05-21,15,Sunday,1,21,141
2023-05-23,30,Tuesday,3,23,143
2023-10-26,45,Thursday,5,26,299
2023-10-28,32,Saturday,7,28,301
2023-10-29,47,Sunday,1,29,302


root
 |-- date: date (nullable = true)
 |-- sales: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_year: integer (nullable = true)



##### 2) Using with current_date()

In [0]:
df_curr = spark.range(1) \
    .withColumn("today", current_date()) \
    .withColumn('day_name', F.date_format(current_date(), 'EEEE')) \
    .withColumn("day_of_week", dayofweek("today")) \
    .withColumn("day_of_month", dayofmonth("today")) \
    .withColumn("day_of_year", dayofyear("today"))

display(df_curr)
df_curr.printSchema()

id,today,day_name,day_of_week,day_of_month,day_of_year
0,2026-03-08,Sunday,1,8,67


root
 |-- id: long (nullable = false)
 |-- today: date (nullable = false)
 |-- day_name: string (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- day_of_month: integer (nullable = false)
 |-- day_of_year: integer (nullable = false)



##### 3) Filtering Based on Day of Week
- `Get only weekend records (Sunday = 1, Saturday = 7)`

In [0]:
df_weekend = df_wmd_01.filter(dayofweek(col("date")).isin(1,7))
display(df_weekend)

date,sales,day_name,day_of_week,day_of_month,day_of_year
2023-01-01,10,Sunday,1,1,1
2023-04-02,20,Sunday,1,2,92
2023-04-08,22,Saturday,7,8,98
2023-04-15,14,Saturday,7,15,105
2023-05-21,15,Sunday,1,21,141
2023-10-28,32,Saturday,7,28,301
2023-10-29,47,Sunday,1,29,302
2023-12-31,50,Sunday,1,31,365


##### 4) Business Logic
- `Add a flag if order is placed in first 110 days of year`.

In [0]:
df_log = df_dow \
    .withColumn('day_name', F.date_format(col("date"), 'EEEE')) \
    .withColumn("day_of_year", dayofyear(col("date"))) \
    .withColumn("early_year_flag", when(dayofyear(col("date")) <= 110, "Y").otherwise("N"))

display(df_log)

date,sales,day_name,day_of_year,early_year_flag
2023-04-02,20,Sunday,92,Y
2023-04-08,22,Saturday,98,Y
2023-04-15,14,Saturday,105,Y
2023-04-17,12,Monday,107,Y
2023-05-21,15,Sunday,141,N
2023-05-23,30,Tuesday,143,N
2023-10-26,45,Thursday,299,N
2023-10-28,32,Saturday,301,N
2023-10-29,47,Sunday,302,N
